# 34 (PW) — Train & Serve

**Production workflow, step 3.** Fit an estimator on IRISpark DataFrames, evaluate on a held-out split, persist the model, and batch-score new rows. Grounded in `ml_scope.md` and `docs/api_reference.md`.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Build a small training frame

`y = 2x + 1` plus noise — an easy sanity check for the linear fit.

In [ ]:
import random
import pandas as pd

random.seed(34)
rows = []
for i in range(60):
    x = random.uniform(-3, 3)
    rows.append({"x": x, "y": 2 * x + 1 + random.uniform(-0.5, 0.5)})
df = session.createDataFrame(pd.DataFrame(rows))
print("rows:", df.count())

## 2. Train/test split (leak-free)

`randomSplit` with a fixed seed; materialize splits so later `transform` subqueries stay stable.

In [ ]:
train, test = df.randomSplit([0.8, 0.2], seed=7)
train.write.mode("overwrite").saveAsTable("pw_train")
test.write.mode("overwrite").saveAsTable("pw_test")
train = session.table("pw_train")
test = session.table("pw_test")
print("train:", train.count(), "| test:", test.count())

## 3. Fit

`LinearRegression` fits closed-form normal equations in numpy; scoring pushes back to SQL.

In [ ]:
from irispark.ml.regression import LinearRegression

lr = LinearRegression(featuresCol=["x"], labelCol="y")
model = lr.fit(train)
print("backend:", model.backend)
print("intercept:", round(model.intercept, 3), "| coef:", [round(c, 3) for c in model.coefficients])

## 4. Evaluate on the held-out split

MAE / MSE / RMSE / R² via `RegressionEvaluator`.

In [ ]:
from irispark.ml.evaluation import RegressionEvaluator

pred = model.transform(test)
pred.select("x", "y", "prediction").show(5)
for m in ("mae", "mse", "rmse", "r2"):
    ev = RegressionEvaluator(predictionCol="prediction", labelCol="y", metricName=m)
    print(f"{m}: {ev.evaluate(pred):.4f}")

## 5. Persist the model

Save to IRIS (`IRISML.Model`), then reload by name — the standard deploy step.

In [ ]:
from irispark.ml import save, load_by_name, list_models

mid = model.save("pw_default_model", session=session)
print("saved model id:", mid)
loaded = load_by_name("pw_default_model", session)
print("coeffs match:", loaded.coefficients == model.coefficients)
print("models on server:", list_models(session) if callable(list_models) else "list_models available")

## 6. Serve — batch-score new rows

Score a fresh table without moving data to Python: `transform` pushes the learned coefficients into SQL.

In [ ]:
import pandas as pd

novos = session.createDataFrame(pd.DataFrame({"x": [10.0, -5.0, 0.0]}))
scored = loaded.transform(novos)
scored.select("x", "prediction").show()

scored.write.mode("overwrite").saveAsTable("pw_scored")
print("scored rows persisted:", session.table("pw_scored").count())

## 7. Cleanup

In [ ]:
from irispark.ml import delete_model

delete_model(mid, session) if callable(delete_model) else None
for t in ("pw_train", "pw_test", "pw_scored"):
    session.sql(f"DROP TABLE IF EXISTS {t}")
print("cleaned up")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")